# Generate some text prompts

In [127]:
# !pip install --upgrade pip
#!pip install numpy pandas rich


In [128]:
from typing import List, Sequence, Tuple

from rich.console import Console
from rich.table import Table

import numpy as np
from numpy.random import default_rng, Generator

rng: Generator = default_rng(seed=123)


In [129]:
with open("list_of_actions.txt", "r", encoding="utf-8") as actions_file:
    actions = actions_file.read().splitlines()
actions = np.asarray(list(filter(None, actions)))  # filter empty ones
actions


array(['walking', 'running', 'jogging', 'walking backwards',
       'walking in circles', 'waving with their right hand',
       'waving with their left hand', 'clapping', 'shaking a hand',
       'jumping', 'jumping on their left leg',
       'jumping on their right leg', 'sitting down',
       'making swimming motions', 'picking something up from the floor',
       'picking up something very heavy', 'grabbing something overhead',
       'breakdancing', 'dancing a waltz', 'making a hand stand',
       'making burpees', 'making push-ups', 'making sit-ups',
       'exercising'], dtype='<U35')

In [130]:
with open("list_of_templates.txt", "r", encoding="utf-8") as templates_file:
    templates = templates_file.read().splitlines()
templates = np.asarray(templates)
templates


array(['A person {activity}.', 'A person is {activity}.',
       'Someone {activity}.', 'Someone is {activity}.',
       'We see someone who is {activity}.', 'The subject is {activity}.'],
      dtype='<U33')

In [131]:
def sample_actions(number: int, distinct: bool = False) -> Sequence[str]:
    return rng.choice(actions, size=number, replace=not distinct, shuffle=True)


list(sample_actions(4))


['walking',
 'grabbing something overhead',
 'picking something up from the floor',
 'running']

In [132]:
def sample_templates(number: int, distinct: bool = False) -> Sequence[str]:
    return rng.choice(templates, size=number, replace=not distinct, shuffle=True)


list(sample_templates(4))


['The subject is {activity}.',
 'A person is {activity}.',
 'A person is {activity}.',
 'A person is {activity}.']

In [133]:
def sample_timestaps(
    num_steps: int,
    entire_time_length: float,
    min_fraction_of_time_per_step: float,
) -> np.ndarray:
    """
    The difference is sampled uniformly.

    Always starts with zero and ends with something strictly less than ``entire_time_length``.

    Args:
        min_fraction_of_time_per_action:
            The minimum fraction of time that each action should receive,
            relative to a single actions time when making each action as long as all others.
            Sof if 5 actions are drawn for a 10 seconds time series, each one would get ``entire_time_length / 5 == 2.0`` seconds.
            If ``min_fraction_of_time_per_action == 0.25``, then each action would have at least
            ``2.0 * min_fraction_of_time_per_action == 0.5`` seconds length.
    """
    assert num_steps > 1

    # Make random numbers that sum up to 1
    random_part = rng.random(num_steps)  # this is uniform sampling
    random_part /= random_part.sum()

    # Make a vector of all the minimum time lengths
    assert min_fraction_of_time_per_step > 0.0
    mins = np.full((num_steps,), (1 / num_steps) * min_fraction_of_time_per_step)

    # Combine them
    normalized_time_steps = mins + (1 - min_fraction_of_time_per_step) * random_part
    timestamps = (normalized_time_steps * entire_time_length).cumsum()

    # We want to start with zero instead of ending with entire_time_length
    return timestamps - timestamps[0]


sample_timestaps(4, entire_time_length=10, min_fraction_of_time_per_step=0.2)


array([0.        , 3.46937577, 7.34553242, 8.85681042])

In [134]:
def sample_prompt_sequence(
    min_actions: int,
    max_actions: int,
    entire_time_length: float,
    min_fraction_of_time_per_step: float = 0.3,
    distinct_actions: bool = False,
    distinct_templates: bool = False,
) -> Sequence[Tuple[float, float, str]]:
    num_actions = rng.integers(min_actions, max_actions, endpoint=True).item()
    actions = sample_actions(num_actions, distinct=distinct_actions)
    templates = sample_templates(num_actions, distinct=distinct_templates)
    text_prompts = (
        template.format(activity=action) for template, action in zip(templates, actions)
    )

    time_stamps_start = sample_timestaps(
        num_actions,
        entire_time_length=entire_time_length,
        min_fraction_of_time_per_step=min_fraction_of_time_per_step,
    )
    # make them non-overlapping
    time_stamps_end = np.concatenate((time_stamps_start[1:], (entire_time_length,)))

    return list(zip(time_stamps_start, time_stamps_end, text_prompts))


sample_prompt_sequence(3, 4, entire_time_length=10)


[(0.0, 2.815715150912469, 'Someone is making a hand stand.'),
 (2.815715150912469, 5.320718317766453, 'A person is making burpees.'),
 (5.320718317766453, 8.654459859926314, 'A person is making push-ups.'),
 (8.654459859926314, 10.0, 'A person is walking.')]

In [135]:
console = Console()


def print_prompt_sequence(prompt_sequence) -> None:
    table = Table(title=f"Data instance {i:06d}", width=100)
    table.add_column("start", justify="right", style="cyan", ratio=0.1)
    table.add_column("end", justify="right", style="magenta", ratio=0.1)
    table.add_column("prompt sequence", style="green", ratio=0.8)
    for timestamp_start, timestamp_end, prompt in prompt_sequence:
        table.add_row(f"{timestamp_start:.4f}", f"{timestamp_end:.4f}", prompt)
    console.print(table)


In [136]:
num_samples = 3

console = Console()

for i in range(num_samples):
    prompt_sequence = sample_prompt_sequence(
        min_actions=2,
        max_actions=5,
        entire_time_length=18.0,
        min_fraction_of_time_per_step=0.3,
    )
    print_prompt_sequence(prompt_sequence)


                                        Data instance 000000                                        
┏━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃    start ┃      end ┃ prompt sequence                                                            ┃
┡━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│   0.0000 │   5.2735 │ Someone is breakdancing.                                                   │
│   5.2735 │   7.6801 │ We see someone who is waving with their right hand.                        │
│   7.6801 │   8.8673 │ A person is making burpees.                                                │
│   8.8673 │  13.3377 │ A person is making a hand stand.                                           │
│  13.3377 │  18.0000 │ A person waving with their right hand.                                     │
└──────────┴──────────┴────────────────────────────────────────────────────────────────────────────┘

                                        Data instance 000001                                        
┏━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃    start ┃      end ┃ prompt sequence                                                            ┃
┡━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│   0.0000 │   8.8146 │ The subject is breakdancing.                                               │
│   8.8146 │  18.0000 │ A person is waving with their left hand.                                   │
└──────────┴──────────┴────────────────────────────────────────────────────────────────────────────┘

                                        Data instance 000002                                        
┏━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃    start ┃      end ┃ prompt sequence                                                            ┃
┡━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│   0.0000 │   5.7304 │ A person is jumping on their right leg.                                    │
│   5.7304 │   8.5964 │ We see someone who is waving with their right hand.                        │
│   8.5964 │  12.6147 │ Someone is walking in circles.                                             │
│  12.6147 │  18.0000 │ A person is making burpees.                                                │
└──────────┴──────────┴────────────────────────────────────────────────────────────────────────────┘